In [1]:
# ============================================================
# 06_AURORA_allocation_stress_tests_and_paper_figures.ipynb
# AURORA-TWETF Allocation Stress Tests and Paper Figures
#
# Purpose:
# 1. Load Notebook 05 outputs.
# 2. Reconstruct allocation inputs from Notebook 04B and ETF returns.
# 3. Stress-test allocation design choices:
#    - cash gating intensity
#    - max 00881 semiconductor ETF cap
#    - max ETF cap
#    - 20d/60d probability blend ratio
#    - rebalancing frequency
#    - transaction costs
#    - allocation template aggressiveness
# 4. Compare AURORA policies against passive ETF benchmarks.
# 5. Create paper-ready tables and figures.
# 6. Save validation report and SHA-256 manifest.
#
# Important:
# - Educational/research backtest only.
# - Not personalized financial advice.
# - Test-set stress tests should be interpreted as sensitivity analysis,
#   not as a finalized trading strategy selection.
# ============================================================

from __future__ import annotations

import os
import sys
import json
import math
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# ============================================================
# 0. Colab setup
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped or failed.")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# 1. Paths and global configuration
# ============================================================

PROJECT_CODE = "AURORA_TWETF"
PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

DATA_ROOT = PUBLICATION_ROOT / "data"
PANEL_DIR = DATA_ROOT / "panels"
MODELING_DIR = DATA_ROOT / "modeling"

OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE
TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"
FIGURE_DIR = OUTPUT_ROOT / "figures"

# Notebook 05 completed run.
NOTEBOOK05_RUN_ID = "20260624_004516"
NOTEBOOK05_ROOT = OUTPUT_ROOT / "uncertainty_aware_etf_allocation" / f"run_{NOTEBOOK05_RUN_ID}"

# Notebook 04B registry from Notebook 05.
NOTEBOOK05_INPUT_INDEX = Path(
    "/content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/"
    "ordinal_imbalance_uncertainty/run_20260623_151920/"
    "allocation_inputs_adjusted_before_05/NOTEBOOK05_INPUT_INDEX.csv"
)

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

STRESS_ROOT = OUTPUT_ROOT / "allocation_stress_tests_and_paper_figures"
RUN_ROOT = STRESS_ROOT / f"run_{RUN_ID}"

TABLE_RUN_DIR = RUN_ROOT / "tables"
PLOT_DIR = RUN_ROOT / "plots"
RETURN_DIR = RUN_ROOT / "returns"
WEIGHT_DIR = RUN_ROOT / "weights"
REPORT_RUN_DIR = RUN_ROOT / "reports"
FIGURE_RUN_DIR = RUN_ROOT / "paper_figures"
DIAGNOSTIC_DIR = RUN_ROOT / "diagnostics"

for d in [
    OUTPUT_ROOT,
    TABLE_DIR,
    REPORT_DIR,
    FIGURE_DIR,
    STRESS_ROOT,
    RUN_ROOT,
    TABLE_RUN_DIR,
    PLOT_DIR,
    RETURN_DIR,
    WEIGHT_DIR,
    REPORT_RUN_DIR,
    FIGURE_RUN_DIR,
    DIAGNOSTIC_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("AURORA-TWETF Notebook 06: Allocation Stress Tests and Paper Figures")
print("=" * 80)
print("Timestamp UTC       :", RUN_TIMESTAMP)
print("Run ID              :", RUN_ID)
print("Project root        :", PUBLICATION_ROOT)
print("Notebook 05 root    :", NOTEBOOK05_ROOT)
print("Notebook 05 registry:", NOTEBOOK05_INPUT_INDEX)
print("Run root            :", RUN_ROOT)
print("=" * 80)

if not NOTEBOOK05_ROOT.exists():
    raise FileNotFoundError(
        f"Notebook 05 run root not found:\n{NOTEBOOK05_ROOT}\n"
        "Please confirm NOTEBOOK05_RUN_ID."
    )

if not NOTEBOOK05_INPUT_INDEX.exists():
    raise FileNotFoundError(
        f"Notebook 05 input index not found:\n{NOTEBOOK05_INPUT_INDEX}\n"
        "Please run Notebook 04B first."
    )

# ============================================================
# 2. Core configuration
# ============================================================

ETF_UNIVERSE = ["0050", "006208", "00692", "00881"]
CASH_COL = "CASH"
CLASS_LABELS = [0, 1, 2, 3, 4]

REGIME_LABEL_DEFINITION = {
    0: "Strong Bear",
    1: "Bear",
    2: "Neutral",
    3: "Bull",
    4: "Strong Bull",
}

ANNUALIZATION_DAYS = 252
INITIAL_CAPITAL = 1.0

# Baseline Notebook 05 assumptions.
BASELINE_CONFIG = {
    "rebalance_frequency": "quarterly",
    "transaction_cost_rate": 0.0010,
    "alpha_20d": 0.60,
    "alpha_60d": 0.40,
    "min_confidence_for_full_risk": 0.55,
    "max_confidence_for_min_risk": 0.20,
    "uncertainty_cash_boost_max": 0.20,
    "max_etf_weight": 0.45,
    "max_00881_weight": 0.35,
    "max_cash_weight": 0.50,
    "min_cash_weight": 0.00,
    "template_variant": "baseline",
}

POLICIES_TO_STRESS = [
    "P1_validation_selected",
    "P2_test_robust_reference",
    "P3_conservative_ordinal_60d",
    "P4_calibrated_linear_60d",
    "P5_notebook04_probability_ensemble",
    "P6_custom_robust_60d_weighted_ensemble",
]

PRIMARY_POLICY = "P4_calibrated_linear_60d"

P6_FALLBACK_20D_POLICY = "P2_test_robust_reference"

# Notebook 05 class-conditioned allocation templates.
BASELINE_CLASS_WEIGHT_TEMPLATES = {
    0: {"0050": 0.15, "006208": 0.25, "00692": 0.25, "00881": 0.00, "CASH": 0.35},
    1: {"0050": 0.25, "006208": 0.30, "00692": 0.30, "00881": 0.05, "CASH": 0.10},
    2: {"0050": 0.30, "006208": 0.30, "00692": 0.25, "00881": 0.15, "CASH": 0.00},
    3: {"0050": 0.30, "006208": 0.25, "00692": 0.20, "00881": 0.25, "CASH": 0.00},
    4: {"0050": 0.25, "006208": 0.20, "00692": 0.15, "00881": 0.40, "CASH": 0.00},
}

# More aggressive: lower cash in bear regimes and higher 00881 in bullish regimes.
AGGRESSIVE_CLASS_WEIGHT_TEMPLATES = {
    0: {"0050": 0.20, "006208": 0.30, "00692": 0.25, "00881": 0.05, "CASH": 0.20},
    1: {"0050": 0.28, "006208": 0.30, "00692": 0.27, "00881": 0.10, "CASH": 0.05},
    2: {"0050": 0.30, "006208": 0.28, "00692": 0.22, "00881": 0.20, "CASH": 0.00},
    3: {"0050": 0.28, "006208": 0.22, "00692": 0.15, "00881": 0.35, "CASH": 0.00},
    4: {"0050": 0.22, "006208": 0.18, "00692": 0.10, "00881": 0.50, "CASH": 0.00},
}

# More defensive: higher cash in bear/neutral and lower 00881.
DEFENSIVE_CLASS_WEIGHT_TEMPLATES = {
    0: {"0050": 0.10, "006208": 0.20, "00692": 0.25, "00881": 0.00, "CASH": 0.45},
    1: {"0050": 0.20, "006208": 0.25, "00692": 0.30, "00881": 0.00, "CASH": 0.25},
    2: {"0050": 0.30, "006208": 0.30, "00692": 0.25, "00881": 0.05, "CASH": 0.10},
    3: {"0050": 0.32, "006208": 0.28, "00692": 0.25, "00881": 0.15, "CASH": 0.00},
    4: {"0050": 0.30, "006208": 0.25, "00692": 0.20, "00881": 0.25, "CASH": 0.00},
}

TEMPLATE_VARIANTS = {
    "baseline": BASELINE_CLASS_WEIGHT_TEMPLATES,
    "aggressive": AGGRESSIVE_CLASS_WEIGHT_TEMPLATES,
    "defensive": DEFENSIVE_CLASS_WEIGHT_TEMPLATES,
}

# Stress test scenarios.
STRESS_SCENARIOS = [
    # Baseline.
    {"scenario_name": "S00_baseline_quarterly_10bps", **BASELINE_CONFIG},

    # Transaction cost sensitivity.
    {**BASELINE_CONFIG, "scenario_name": "S01_zero_transaction_cost", "transaction_cost_rate": 0.0000},
    {**BASELINE_CONFIG, "scenario_name": "S02_high_transaction_cost_25bps", "transaction_cost_rate": 0.0025},

    # Rebalancing sensitivity.
    {**BASELINE_CONFIG, "scenario_name": "S03_monthly_rebalance", "rebalance_frequency": "monthly"},
    {**BASELINE_CONFIG, "scenario_name": "S04_quarterly_rebalance", "rebalance_frequency": "quarterly"},

    # Cash gating sensitivity.
    {**BASELINE_CONFIG, "scenario_name": "S05_no_uncertainty_cash_boost", "uncertainty_cash_boost_max": 0.00},
    {**BASELINE_CONFIG, "scenario_name": "S06_low_cash_boost_10pct", "uncertainty_cash_boost_max": 0.10},
    {**BASELINE_CONFIG, "scenario_name": "S07_high_cash_boost_30pct", "uncertainty_cash_boost_max": 0.30},

    # 00881 / semiconductor cap sensitivity.
    {**BASELINE_CONFIG, "scenario_name": "S08_low_00881_cap_25pct", "max_00881_weight": 0.25},
    {**BASELINE_CONFIG, "scenario_name": "S09_baseline_00881_cap_35pct", "max_00881_weight": 0.35},
    {**BASELINE_CONFIG, "scenario_name": "S10_high_00881_cap_50pct", "max_00881_weight": 0.50},

    # ETF cap sensitivity.
    {**BASELINE_CONFIG, "scenario_name": "S11_max_etf_cap_35pct", "max_etf_weight": 0.35},
    {**BASELINE_CONFIG, "scenario_name": "S12_max_etf_cap_60pct", "max_etf_weight": 0.60},

    # Blend ratio sensitivity.
    {**BASELINE_CONFIG, "scenario_name": "S13_more_20d_alpha_80_20", "alpha_20d": 0.80, "alpha_60d": 0.20},
    {**BASELINE_CONFIG, "scenario_name": "S14_equal_alpha_50_50", "alpha_20d": 0.50, "alpha_60d": 0.50},
    {**BASELINE_CONFIG, "scenario_name": "S15_more_60d_alpha_30_70", "alpha_20d": 0.30, "alpha_60d": 0.70},

    # Template aggressiveness.
    {**BASELINE_CONFIG, "scenario_name": "S16_aggressive_templates", "template_variant": "aggressive", "max_00881_weight": 0.50},
    {**BASELINE_CONFIG, "scenario_name": "S17_defensive_templates", "template_variant": "defensive"},

    # Combined less defensive stress.
    {
        **BASELINE_CONFIG,
        "scenario_name": "S18_less_defensive_combined",
        "template_variant": "aggressive",
        "uncertainty_cash_boost_max": 0.05,
        "max_00881_weight": 0.50,
        "max_etf_weight": 0.60,
        "alpha_20d": 0.80,
        "alpha_60d": 0.20,
    },
]

# ============================================================
# 3. Utility functions
# ============================================================

def save_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def make_file_manifest(root):
    root = Path(root)
    rows = []

    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc,
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })

    return pd.DataFrame(rows)

def safe_name(x):
    return (
        str(x)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(" ", "_")
    )

def read_table_auto(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    if path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
        df = df.set_index("date")
    else:
        try:
            df.index = pd.to_datetime(df.index)
        except Exception:
            pass

    return df.sort_index()

def find_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

def clean_symbol_name(x):
    x = str(x)
    x = x.replace(".TW", "")
    x = x.replace(".TWO", "")
    x = x.replace("TW_", "")
    return x

def load_etf_return_panel():
    candidates = [
        PANEL_DIR / "AURORA_etf_return_panel.parquet",
        PANEL_DIR / "AURORA_etf_returns_panel.parquet",
        PANEL_DIR / "AURORA_return_panel.parquet",
        MODELING_DIR / "AURORA_etf_return_panel.parquet",
    ]

    path = find_first_existing(candidates)

    if path is None:
        raise FileNotFoundError(
            "Could not find ETF return panel. Tried:\n"
            + "\n".join(str(p) for p in candidates)
        )

    df = pd.read_parquet(path)
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    df = df.rename(columns={c: clean_symbol_name(c) for c in df.columns})

    missing = [s for s in ETF_UNIVERSE if s not in df.columns]
    if missing:
        raise ValueError(
            f"ETF return panel found at {path}, but missing ETF columns: {missing}\n"
            f"Available columns: {list(df.columns)}"
        )

    df = df[ETF_UNIVERSE].copy()
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    print("Loaded ETF return panel:", path)
    print("ETF return shape       :", df.shape)
    print("ETF return date range  :", df.index.min().date(), "to", df.index.max().date())

    return df, path

def proba_cols_from_df(df):
    return [c for c in df.columns if str(c).startswith("proba_class_")]

def ensure_valid_proba_matrix(p):
    p = np.asarray(p, dtype=float)
    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    p[p < 0] = 0.0

    row_sums = p.sum(axis=1, keepdims=True)
    zero_rows = row_sums[:, 0] <= 0

    if np.any(zero_rows):
        p[zero_rows, :] = 1.0 / p.shape[1]
        row_sums = p.sum(axis=1, keepdims=True)

    return p / row_sums

def probability_features(proba_df):
    proba_cols = [f"proba_class_{i}" for i in CLASS_LABELS]

    missing = [c for c in proba_cols if c not in proba_df.columns]
    if missing:
        raise ValueError(f"Missing probability columns: {missing}")

    p = ensure_valid_proba_matrix(proba_df[proba_cols].values)

    classes = np.asarray(CLASS_LABELS, dtype=float)
    expected_class = p @ classes
    entropy = -np.sum(np.clip(p, 1e-12, 1.0) * np.log(np.clip(p, 1e-12, 1.0)), axis=1)
    normalized_entropy = entropy / np.log(len(CLASS_LABELS))
    sorted_p = np.sort(p, axis=1)
    margin = sorted_p[:, -1] - sorted_p[:, -2]
    ordinal_var = (p @ (classes ** 2)) - expected_class ** 2
    confidence = 1.0 - normalized_entropy

    out = pd.DataFrame(index=proba_df.index)
    out["expected_class"] = expected_class
    out["entropy"] = entropy
    out["normalized_entropy"] = normalized_entropy
    out["confidence_score"] = confidence
    out["probability_margin"] = margin
    out["ordinal_variance"] = ordinal_var
    out["p_bearish"] = p[:, 0] + p[:, 1]
    out["p_neutral"] = p[:, 2]
    out["p_bullish"] = p[:, 3] + p[:, 4]

    for i, c in enumerate(CLASS_LABELS):
        out[f"proba_class_{c}"] = p[:, i]

    return out

def class_template_matrix(template_dict):
    cols = ETF_UNIVERSE + [CASH_COL]
    mat = np.zeros((len(CLASS_LABELS), len(cols)), dtype=float)

    for c in CLASS_LABELS:
        template = template_dict[c]
        for j, col in enumerate(cols):
            mat[c, j] = float(template.get(col, 0.0))

    row_sums = mat.sum(axis=1, keepdims=True)
    row_sums[row_sums <= 0] = 1.0

    mat = mat / row_sums
    return mat, cols

def apply_weight_constraints(weight_vec, cols, config):
    w = pd.Series(weight_vec, index=cols, dtype=float)
    w = w.clip(lower=0.0)

    max_etf_weight = float(config["max_etf_weight"])
    max_00881_weight = float(config["max_00881_weight"])
    max_cash_weight = float(config["max_cash_weight"])
    min_cash_weight = float(config["min_cash_weight"])

    if CASH_COL in w.index:
        w[CASH_COL] = min(max(w[CASH_COL], min_cash_weight), max_cash_weight)

    for etf in ETF_UNIVERSE:
        if etf in w.index:
            cap = max_etf_weight
            if etf == "00881":
                cap = min(cap, max_00881_weight)
            w[etf] = min(w[etf], cap)

    total = w.sum()
    if total <= 0:
        neutral = pd.Series(BASELINE_CLASS_WEIGHT_TEMPLATES[2], dtype=float).reindex(cols).fillna(0.0)
        w = neutral.copy()
        total = w.sum()

    w = w / total

    for _ in range(10):
        excess = 0.0
        capped = []

        for etf in ETF_UNIVERSE:
            cap = max_etf_weight
            if etf == "00881":
                cap = min(cap, max_00881_weight)

            if w.get(etf, 0.0) > cap:
                excess += w[etf] - cap
                w[etf] = cap
                capped.append(etf)

        if CASH_COL in w.index and w[CASH_COL] > max_cash_weight:
            excess += w[CASH_COL] - max_cash_weight
            w[CASH_COL] = max_cash_weight
            capped.append(CASH_COL)

        if excess <= 1e-12:
            break

        eligible = [c for c in cols if c not in capped]

        if not eligible:
            break

        eligible_sum = w[eligible].sum()
        if eligible_sum <= 0:
            w[eligible] += excess / len(eligible)
        else:
            w[eligible] += excess * (w[eligible] / eligible_sum)

    w = w.clip(lower=0.0)
    total = w.sum()

    if total <= 0:
        neutral = pd.Series(BASELINE_CLASS_WEIGHT_TEMPLATES[2], dtype=float).reindex(cols).fillna(0.0)
        w = neutral / neutral.sum()
    else:
        w = w / total

    return w

def proba_to_template_weights(proba_df, template_dict, config):
    proba_cols = [f"proba_class_{i}" for i in CLASS_LABELS]
    p = ensure_valid_proba_matrix(proba_df[proba_cols].values)

    template_mat, cols = class_template_matrix(template_dict)
    raw_w = p @ template_mat

    weight_df = pd.DataFrame(raw_w, index=proba_df.index, columns=cols)

    constrained = []
    for _, row in weight_df.iterrows():
        constrained.append(apply_weight_constraints(row.values, cols, config).values)

    return pd.DataFrame(constrained, index=weight_df.index, columns=cols)

def blend_weights(w_20, w_60, config):
    cols = ETF_UNIVERSE + [CASH_COL]

    alpha_20d = float(config["alpha_20d"])
    alpha_60d = float(config["alpha_60d"])

    total_alpha = alpha_20d + alpha_60d
    if total_alpha <= 0:
        alpha_20d, alpha_60d = 0.5, 0.5
    else:
        alpha_20d, alpha_60d = alpha_20d / total_alpha, alpha_60d / total_alpha

    w_20 = w_20.reindex(columns=cols).fillna(0.0)
    w_60 = w_60.reindex(columns=cols).fillna(0.0)

    common_idx = w_20.index.intersection(w_60.index)
    out = alpha_20d * w_20.loc[common_idx] + alpha_60d * w_60.loc[common_idx]

    constrained = []
    for _, row in out.iterrows():
        constrained.append(apply_weight_constraints(row.values, cols, config).values)

    return pd.DataFrame(constrained, index=out.index, columns=cols)

def apply_uncertainty_gating(weight_df, feature_df, template_dict, config):
    cols = ETF_UNIVERSE + [CASH_COL]

    neutral = pd.Series(template_dict[2], dtype=float).reindex(cols).fillna(0.0)
    neutral = apply_weight_constraints(neutral.values, cols, config)

    min_conf = float(config["min_confidence_for_full_risk"])
    max_conf = float(config["max_confidence_for_min_risk"])
    cash_boost_max = float(config["uncertainty_cash_boost_max"])

    out_rows = []

    for dt, row in weight_df.iterrows():
        if dt not in feature_df.index:
            out_rows.append(apply_weight_constraints(row.values, cols, config).values)
            continue

        confidence = float(feature_df.loc[dt, "combined_confidence"])
        p_bearish = float(feature_df.loc[dt, "combined_p_bearish"])
        ord_var = float(feature_df.loc[dt, "combined_ordinal_variance"])

        denom = min_conf - max_conf
        if denom <= 0:
            risk_scale = 1.0
        else:
            risk_scale = (confidence - max_conf) / denom
            risk_scale = float(np.clip(risk_scale, 0.0, 1.0))

        signal_w = pd.Series(row, index=cols, dtype=float)
        gated = risk_scale * signal_w + (1.0 - risk_scale) * neutral

        uncertainty = 1.0 - confidence
        cash_boost = cash_boost_max * uncertainty * min(1.0, p_bearish + 0.25 * ord_var)
        cash_boost = float(np.clip(cash_boost, 0.0, cash_boost_max))

        if CASH_COL in gated.index and cash_boost > 0:
            etf_cols = [c for c in ETF_UNIVERSE if c in gated.index]
            reduce_pool = gated[etf_cols].sum()

            if reduce_pool > 0:
                reduction_ratio = cash_boost / max(reduce_pool + cash_boost, 1e-12)
                gated[etf_cols] = gated[etf_cols] * (1.0 - reduction_ratio)
                gated[CASH_COL] = gated[CASH_COL] + cash_boost

        gated = apply_weight_constraints(gated.values, cols, config)
        out_rows.append(gated.values)

    return pd.DataFrame(out_rows, index=weight_df.index, columns=cols)

def get_rebalance_dates(index, frequency):
    idx = pd.DatetimeIndex(index).sort_values()

    if frequency == "monthly":
        groups = pd.Series(idx, index=idx).groupby([idx.year, idx.month])
    elif frequency == "quarterly":
        groups = pd.Series(idx, index=idx).groupby([idx.year, idx.quarter])
    elif frequency == "weekly":
        iso = idx.isocalendar()
        groups = pd.Series(idx, index=idx).groupby([iso.year, iso.week])
    else:
        raise ValueError(f"Unsupported rebalance frequency: {frequency}")

    dates = []
    for _, values in groups:
        dates.append(values.iloc[0])

    return pd.DatetimeIndex(dates)

def expand_rebalance_weights_to_daily(signal_weight_df, daily_index, rebalance_dates):
    cols = signal_weight_df.columns.tolist()
    daily_weights = pd.DataFrame(index=daily_index, columns=cols, dtype=float)

    signal_idx = pd.DatetimeIndex(signal_weight_df.index).sort_values()

    for i, reb_date in enumerate(rebalance_dates):
        effective_start = reb_date

        if i + 1 < len(rebalance_dates):
            effective_end = rebalance_dates[i + 1]
            period_idx = daily_index[(daily_index >= effective_start) & (daily_index < effective_end)]
        else:
            period_idx = daily_index[daily_index >= effective_start]

        prior_signals = signal_idx[signal_idx < reb_date]

        if len(prior_signals) == 0:
            signal_date = signal_idx[0]
        else:
            signal_date = prior_signals[-1]

        current_w = signal_weight_df.loc[signal_date].copy()
        daily_weights.loc[period_idx, :] = current_w.values

    daily_weights = daily_weights.ffill().bfill()
    return daily_weights

def compute_turnover(daily_weights, rebalance_dates):
    turnover = pd.Series(0.0, index=daily_weights.index)

    prev_w = None

    for dt in rebalance_dates:
        if dt not in daily_weights.index:
            continue

        w = daily_weights.loc[dt]

        if prev_w is None:
            turnover.loc[dt] = w.drop(labels=[CASH_COL], errors="ignore").abs().sum()
        else:
            turnover.loc[dt] = (w - prev_w).abs().sum() / 2.0

        prev_w = w

    return turnover

def backtest_policy(policy_name, signal_weight_df, etf_returns, config):
    cols = ETF_UNIVERSE + [CASH_COL]

    returns = etf_returns.copy()
    returns[CASH_COL] = 0.0

    common_index = returns.index.intersection(signal_weight_df.index)
    returns = returns.loc[common_index].copy()
    signal_weight_df = signal_weight_df.loc[common_index].copy()

    rebalance_dates = get_rebalance_dates(returns.index, config["rebalance_frequency"])

    daily_weights = expand_rebalance_weights_to_daily(
        signal_weight_df=signal_weight_df,
        daily_index=returns.index,
        rebalance_dates=rebalance_dates,
    )

    daily_weights = daily_weights.reindex(columns=cols).fillna(0.0)

    gross_return = (daily_weights[cols] * returns[cols]).sum(axis=1)

    turnover = compute_turnover(daily_weights, rebalance_dates)
    transaction_cost = turnover * float(config["transaction_cost_rate"])
    net_return = gross_return - transaction_cost

    equity = (1.0 + net_return).cumprod() * INITIAL_CAPITAL

    out = pd.DataFrame(index=returns.index)
    out["policy_name"] = policy_name
    out["gross_return"] = gross_return
    out["turnover"] = turnover
    out["transaction_cost"] = transaction_cost
    out["net_return"] = net_return
    out["equity"] = equity
    out["drawdown"] = equity / equity.cummax() - 1.0
    out["is_rebalance_date"] = out.index.isin(rebalance_dates)

    return out, daily_weights

def performance_metrics(return_df):
    r = return_df["net_return"].astype(float).copy()
    equity = return_df["equity"].astype(float).copy()
    drawdown = return_df["drawdown"].astype(float).copy()

    n = len(r)
    if n == 0:
        return {}

    total_return = float(equity.iloc[-1] / equity.iloc[0] - 1.0) if equity.iloc[0] != 0 else np.nan
    annual_return = float((1.0 + total_return) ** (ANNUALIZATION_DAYS / max(n, 1)) - 1.0)
    annual_vol = float(r.std(ddof=1) * np.sqrt(ANNUALIZATION_DAYS)) if n > 1 else np.nan

    sharpe = annual_return / annual_vol if annual_vol and annual_vol > 0 else np.nan

    downside = r[r < 0]
    downside_vol = float(downside.std(ddof=1) * np.sqrt(ANNUALIZATION_DAYS)) if len(downside) > 1 else np.nan
    sortino = annual_return / downside_vol if downside_vol and downside_vol > 0 else np.nan

    max_drawdown = float(drawdown.min())
    calmar = annual_return / abs(max_drawdown) if max_drawdown < 0 else np.nan

    return {
        "n_days": int(n),
        "start_date": str(r.index.min().date()),
        "end_date": str(r.index.max().date()),
        "total_return": total_return,
        "annual_return": annual_return,
        "annual_volatility": annual_vol,
        "sharpe_ratio": sharpe,
        "sortino_ratio": sortino,
        "max_drawdown": max_drawdown,
        "calmar_ratio": calmar,
        "hit_rate": float((r > 0).mean()),
        "avg_daily_return": float(r.mean()),
        "avg_turnover": float(return_df["turnover"].mean()),
        "total_turnover": float(return_df["turnover"].sum()),
        "total_transaction_cost": float(return_df["transaction_cost"].sum()),
        "final_equity": float(equity.iloc[-1]),
    }

def infer_test_dates(policy_signal_features):
    for _, feature_df in policy_signal_features.items():
        if "split_20d" in feature_df.columns and "split_60d" in feature_df.columns:
            mask = (feature_df["split_20d"] == "test") | (feature_df["split_60d"] == "test")
            idx = pd.DatetimeIndex(feature_df.index[mask])
            if len(idx) > 0:
                return idx
    return None

def rebase_return_df(return_df):
    out = return_df.copy()
    out["equity"] = (1.0 + out["net_return"]).cumprod()
    out["drawdown"] = out["equity"] / out["equity"].cummax() - 1.0
    return out

# ============================================================
# 4. Plotting functions
# ============================================================

def plot_equity_curves(all_returns_df, path, title):
    plt.figure(figsize=(12, 6))
    for label, grp in all_returns_df.groupby("series_name"):
        grp = grp.sort_index()
        plt.plot(grp.index, grp["equity"], label=label, linewidth=1.8)

    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel("Equity, initial capital = 1")
    plt.grid(True, alpha=0.3)
    plt.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.savefig(path, dpi=220)
    plt.close()

def plot_drawdowns(all_returns_df, path, title):
    plt.figure(figsize=(12, 6))
    for label, grp in all_returns_df.groupby("series_name"):
        grp = grp.sort_index()
        plt.plot(grp.index, grp["drawdown"], label=label, linewidth=1.5)

    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel("Drawdown")
    plt.grid(True, alpha=0.3)
    plt.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.savefig(path, dpi=220)
    plt.close()

def plot_metric_bar(df, metric, path, title, higher_is_better=True, top_n=None):
    dfp = df.copy()
    dfp = dfp.replace([np.inf, -np.inf], np.nan).dropna(subset=[metric])

    if top_n is not None:
        dfp = dfp.sort_values(metric, ascending=not higher_is_better).head(top_n)
    else:
        dfp = dfp.sort_values(metric, ascending=not higher_is_better)

    plt.figure(figsize=(11, max(4, 0.42 * len(dfp))))
    sns.barplot(data=dfp, y="series_name", x=metric, color="#4C72B0")
    plt.title(title)
    plt.xlabel(metric)
    plt.ylabel("")
    plt.tight_layout()
    plt.savefig(path, dpi=220)
    plt.close()

def plot_scenario_heatmap(df, row_col, col_col, value_col, path, title):
    pivot = df.pivot_table(index=row_col, columns=col_col, values=value_col, aggfunc="mean")

    plt.figure(figsize=(12, max(4, 0.45 * len(pivot))))
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdYlGn", center=pivot.stack().median())
    plt.title(title)
    plt.xlabel(col_col)
    plt.ylabel(row_col)
    plt.tight_layout()
    plt.savefig(path, dpi=220)
    plt.close()

def plot_sensitivity_line(df, x_col, y_col, hue_col, path, title):
    dfp = df.copy()
    dfp = dfp.replace([np.inf, -np.inf], np.nan).dropna(subset=[x_col, y_col])

    plt.figure(figsize=(10, 5.5))
    sns.lineplot(data=dfp, x=x_col, y=y_col, hue=hue_col, marker="o")
    plt.title(title)
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(path, dpi=220)
    plt.close()

def plot_weight_summary(weight_summary_df, path, title):
    dfp = weight_summary_df.copy()

    plt.figure(figsize=(12, max(4, 0.35 * len(dfp))))
    sns.barplot(data=dfp, y="series_name", x="mean_total_etf_weight", color="#55A868", label="ETF")
    sns.barplot(data=dfp, y="series_name", x="mean_cash_weight", color="#C44E52", label="Cash")

    plt.title(title)
    plt.xlabel("Average weight")
    plt.ylabel("")
    plt.tight_layout()
    plt.savefig(path, dpi=220)
    plt.close()

# ============================================================
# 5. Load data
# ============================================================

print("\n" + "=" * 80)
print("Step 1: Loading Notebook 04B registry, Notebook 05 outputs, and ETF returns")
print("=" * 80)

input_index_df = pd.read_csv(NOTEBOOK05_INPUT_INDEX)
etf_returns, etf_return_path = load_etf_return_panel()

print("\nNotebook 05 registry:")
print(input_index_df[["policy_name", "target_col", "role", "model_name", "source"]].to_string(index=False))

# Optional Notebook 05 performance table.
notebook05_perf_candidates = [
    TABLE_DIR / f"table_29_allocation_performance_{NOTEBOOK05_RUN_ID}.csv",
    NOTEBOOK05_ROOT / "allocation_performance.csv",
]
notebook05_perf_path = find_first_existing(notebook05_perf_candidates)

if notebook05_perf_path is not None:
    notebook05_perf_df = pd.read_csv(notebook05_perf_path)
    print("\nLoaded Notebook 05 performance table:", notebook05_perf_path)
    print("Notebook 05 performance shape:", notebook05_perf_df.shape)
else:
    notebook05_perf_df = None
    print("\nNotebook 05 performance table not found. Continuing.")

# ============================================================
# 6. Load probability files
# ============================================================

print("\n" + "=" * 80)
print("Step 2: Loading probability files from Notebook 04B")
print("=" * 80)

policy_probability_data = {}

for _, row in input_index_df.iterrows():
    policy_name = row["policy_name"]
    target_col = row["target_col"]
    role = row["role"]
    model_name = row["model_name"]

    proba_path = Path(row["probability_path_parquet"])
    if not proba_path.exists():
        proba_path = Path(row["probability_path_csv"])

    proba_df = read_table_auto(proba_path)

    proba_cols = [f"proba_class_{i}" for i in CLASS_LABELS]
    missing = [c for c in proba_cols if c not in proba_df.columns]
    if missing:
        raise ValueError(f"Missing probability columns in {proba_path}: {missing}")

    key = (policy_name, target_col)
    policy_probability_data[key] = {
        "policy_name": policy_name,
        "target_col": target_col,
        "role": role,
        "model_name": model_name,
        "proba_df": proba_df,
        "path": str(proba_path),
    }

    print(f"Loaded {policy_name} | {target_col} | {model_name} | rows={len(proba_df)}")

# ============================================================
# 7. Build signal weights under arbitrary scenario
# ============================================================

def build_policy_signal_weights(policy_name, config):
    template_variant = config.get("template_variant", "baseline")
    template_dict = TEMPLATE_VARIANTS[template_variant]

    key_20 = (policy_name, "TAIEX_regime_fixed_20d")
    key_60 = (policy_name, "TAIEX_regime_fixed_60d")

    if policy_name == "P6_custom_robust_60d_weighted_ensemble" and key_20 not in policy_probability_data:
        key_20 = (P6_FALLBACK_20D_POLICY, "TAIEX_regime_fixed_20d")

    if key_20 not in policy_probability_data:
        raise KeyError(f"Missing 20d probability data for {policy_name}")

    if key_60 not in policy_probability_data:
        raise KeyError(f"Missing 60d probability data for {policy_name}")

    data_20 = policy_probability_data[key_20]
    data_60 = policy_probability_data[key_60]

    p20 = data_20["proba_df"].copy()
    p60 = data_60["proba_df"].copy()

    common_idx = p20.index.intersection(p60.index).intersection(etf_returns.index)

    if len(common_idx) == 0:
        raise ValueError(f"No common dates for policy {policy_name}")

    p20 = p20.loc[common_idx].copy()
    p60 = p60.loc[common_idx].copy()

    f20 = probability_features(p20)
    f60 = probability_features(p60)

    w20 = proba_to_template_weights(p20, template_dict, config)
    w60 = proba_to_template_weights(p60, template_dict, config)

    raw_blended_w = blend_weights(w20, w60, config)

    alpha_20d = float(config["alpha_20d"])
    alpha_60d = float(config["alpha_60d"])
    total_alpha = alpha_20d + alpha_60d
    if total_alpha <= 0:
        alpha_20d, alpha_60d = 0.5, 0.5
    else:
        alpha_20d, alpha_60d = alpha_20d / total_alpha, alpha_60d / total_alpha

    features = pd.DataFrame(index=common_idx)
    features["policy_name"] = policy_name
    features["scenario_name"] = config["scenario_name"]
    features["split_20d"] = p20["split"].values if "split" in p20.columns else "unknown"
    features["split_60d"] = p60["split"].values if "split" in p60.columns else "unknown"
    features["model_20d"] = data_20["model_name"]
    features["model_60d"] = data_60["model_name"]

    features["expected_class_20d"] = f20["expected_class"]
    features["expected_class_60d"] = f60["expected_class"]
    features["confidence_20d"] = f20["confidence_score"]
    features["confidence_60d"] = f60["confidence_score"]
    features["p_bearish_20d"] = f20["p_bearish"]
    features["p_bearish_60d"] = f60["p_bearish"]
    features["p_bullish_20d"] = f20["p_bullish"]
    features["p_bullish_60d"] = f60["p_bullish"]
    features["ordinal_variance_20d"] = f20["ordinal_variance"]
    features["ordinal_variance_60d"] = f60["ordinal_variance"]

    features["combined_expected_class"] = alpha_20d * f20["expected_class"] + alpha_60d * f60["expected_class"]
    features["combined_confidence"] = alpha_20d * f20["confidence_score"] + alpha_60d * f60["confidence_score"]
    features["combined_p_bearish"] = alpha_20d * f20["p_bearish"] + alpha_60d * f60["p_bearish"]
    features["combined_p_bullish"] = alpha_20d * f20["p_bullish"] + alpha_60d * f60["p_bullish"]
    features["combined_ordinal_variance"] = alpha_20d * f20["ordinal_variance"] + alpha_60d * f60["ordinal_variance"]

    gated_w = apply_uncertainty_gating(raw_blended_w, features, template_dict, config)

    component = {
        "policy_name": policy_name,
        "scenario_name": config["scenario_name"],
        "model_20d": data_20["model_name"],
        "source_policy_20d": key_20[0],
        "model_60d": data_60["model_name"],
        "source_policy_60d": key_60[0],
        "n_common_dates": int(len(common_idx)),
        "date_start": str(common_idx.min().date()),
        "date_end": str(common_idx.max().date()),
    }

    return gated_w, features, component

def make_benchmark_weights(index):
    cols = ETF_UNIVERSE + [CASH_COL]

    out = {}

    b1 = pd.DataFrame(0.0, index=index, columns=cols)
    for etf in ETF_UNIVERSE:
        b1[etf] = 1.0 / len(ETF_UNIVERSE)
    out["B1_equal_weight_all_etfs"] = b1

    b2 = pd.DataFrame(0.0, index=index, columns=cols)
    b2["0050"] = 0.30
    b2["006208"] = 0.30
    b2["00692"] = 0.25
    b2["00881"] = 0.15
    out["B2_equal_weight_constrained"] = b2

    b3 = pd.DataFrame(0.0, index=index, columns=cols)
    b3["0050"] = 1.0
    out["B3_0050_only"] = b3

    return out

# ============================================================
# 8. Run stress tests
# ============================================================

print("\n" + "=" * 80)
print("Step 3: Running stress-test scenarios")
print("=" * 80)

all_metrics = []
all_returns_selected = []
all_weights_selected = []
all_components = []
all_feature_summaries = []

test_dates = None

benchmark_weights = make_benchmark_weights(etf_returns.index)

for scenario in STRESS_SCENARIOS:
    scenario_name = scenario["scenario_name"]
    print(f"\nScenario: {scenario_name}")

    scenario_returns = []
    scenario_weight_summary = []

    # Active policies.
    for policy_name in POLICIES_TO_STRESS:
        stress_policy_name = f"{scenario_name}__{policy_name}"

        weights, features, component = build_policy_signal_weights(policy_name, scenario)
        component.update({k: v for k, v in scenario.items() if k != "scenario_name"})
        all_components.append(component)

        returns_df, daily_weights = backtest_policy(
            policy_name=stress_policy_name,
            signal_weight_df=weights,
            etf_returns=etf_returns,
            config=scenario,
        )

        metrics = performance_metrics(returns_df)
        metrics["run_id"] = RUN_ID
        metrics["scenario_name"] = scenario_name
        metrics["base_policy_name"] = policy_name
        metrics["series_name"] = stress_policy_name
        metrics["series_type"] = "active"
        metrics.update({k: v for k, v in scenario.items() if k != "scenario_name"})

        all_metrics.append(metrics)

        scenario_returns.append(returns_df.assign(series_name=stress_policy_name))

        if scenario_name in ["S00_baseline_quarterly_10bps", "S18_less_defensive_combined"] and policy_name in [
            PRIMARY_POLICY,
            "P1_validation_selected",
            "P5_notebook04_probability_ensemble",
        ]:
            all_returns_selected.append(returns_df.assign(series_name=stress_policy_name))
            wf = daily_weights.copy()
            wf.index.name = "date"
            wf.insert(0, "series_name", stress_policy_name)
            all_weights_selected.append(wf)

            daily_weights.to_parquet(WEIGHT_DIR / f"weights_{safe_name(stress_policy_name)}.parquet")
            returns_df.to_parquet(RETURN_DIR / f"returns_{safe_name(stress_policy_name)}.parquet")

        if test_dates is None:
            mask = (features["split_20d"] == "test") | (features["split_60d"] == "test")
            candidate = pd.DatetimeIndex(features.index[mask])
            if len(candidate) > 0:
                test_dates = candidate

        feature_summary = {
            "run_id": RUN_ID,
            "scenario_name": scenario_name,
            "base_policy_name": policy_name,
            "series_name": stress_policy_name,
            "mean_combined_confidence": float(features["combined_confidence"].mean()),
            "mean_combined_p_bearish": float(features["combined_p_bearish"].mean()),
            "mean_combined_p_bullish": float(features["combined_p_bullish"].mean()),
            "mean_combined_ordinal_variance": float(features["combined_ordinal_variance"].mean()),
            "mean_0050_weight": float(daily_weights["0050"].mean()),
            "mean_006208_weight": float(daily_weights["006208"].mean()),
            "mean_00692_weight": float(daily_weights["00692"].mean()),
            "mean_00881_weight": float(daily_weights["00881"].mean()),
            "mean_cash_weight": float(daily_weights[CASH_COL].mean()),
            "mean_total_etf_weight": float(daily_weights[ETF_UNIVERSE].sum(axis=1).mean()),
        }
        all_feature_summaries.append(feature_summary)

    # Benchmarks under same rebalancing/cost assumptions.
    for benchmark_name, benchmark_w in benchmark_weights.items():
        stress_policy_name = f"{scenario_name}__{benchmark_name}"

        returns_df, daily_weights = backtest_policy(
            policy_name=stress_policy_name,
            signal_weight_df=benchmark_w,
            etf_returns=etf_returns,
            config=scenario,
        )

        metrics = performance_metrics(returns_df)
        metrics["run_id"] = RUN_ID
        metrics["scenario_name"] = scenario_name
        metrics["base_policy_name"] = benchmark_name
        metrics["series_name"] = stress_policy_name
        metrics["series_type"] = "benchmark"
        metrics.update({k: v for k, v in scenario.items() if k != "scenario_name"})

        all_metrics.append(metrics)
        scenario_returns.append(returns_df.assign(series_name=stress_policy_name))

    # Save scenario-level returns only for important scenarios to avoid excessive storage.
    if scenario_name in ["S00_baseline_quarterly_10bps", "S18_less_defensive_combined"]:
        scenario_returns_df = pd.concat(scenario_returns, axis=0)
        scenario_returns_df.to_parquet(RETURN_DIR / f"returns_{safe_name(scenario_name)}_all_policies.parquet")
        scenario_returns_df.to_csv(RETURN_DIR / f"returns_{safe_name(scenario_name)}_all_policies.csv")

metrics_df = pd.DataFrame(all_metrics)
components_df = pd.DataFrame(all_components)
feature_summary_df = pd.DataFrame(all_feature_summaries)

metrics_df.to_csv(TABLE_RUN_DIR / "stress_test_metrics_all.csv", index=False)
metrics_df.to_parquet(TABLE_RUN_DIR / "stress_test_metrics_all.parquet", index=False)
metrics_df.to_csv(TABLE_DIR / f"table_33_stress_test_metrics_all_{RUN_ID}.csv", index=False)

components_df.to_csv(TABLE_RUN_DIR / "stress_test_policy_components.csv", index=False)
components_df.to_csv(TABLE_DIR / f"table_34_stress_test_policy_components_{RUN_ID}.csv", index=False)

feature_summary_df.to_csv(TABLE_RUN_DIR / "stress_test_feature_weight_summary.csv", index=False)
feature_summary_df.to_csv(TABLE_DIR / f"table_35_stress_test_feature_weight_summary_{RUN_ID}.csv", index=False)

print("\nStress-test metrics shape:", metrics_df.shape)
print("Feature summary shape    :", feature_summary_df.shape)

# ============================================================
# 9. Test-period metrics
# ============================================================

print("\n" + "=" * 80)
print("Step 4: Computing test-period stress-test metrics")
print("=" * 80)

if test_dates is None or len(test_dates) == 0:
    print("Could not infer test dates from probability splits. Using full return index.")
    test_dates = etf_returns.index

test_metrics = []

for scenario in STRESS_SCENARIOS:
    scenario_name = scenario["scenario_name"]

    # Recompute only for all policies, restricted to test dates.
    for policy_name in POLICIES_TO_STRESS:
        stress_policy_name = f"{scenario_name}__{policy_name}"

        weights, _, _ = build_policy_signal_weights(policy_name, scenario)
        returns_df, _ = backtest_policy(
            policy_name=stress_policy_name,
            signal_weight_df=weights,
            etf_returns=etf_returns,
            config=scenario,
        )

        grp_test = returns_df.loc[returns_df.index.intersection(test_dates)].copy()
        if grp_test.empty:
            continue

        grp_test = rebase_return_df(grp_test)
        tm = performance_metrics(grp_test)
        tm["run_id"] = RUN_ID
        tm["scenario_name"] = scenario_name
        tm["base_policy_name"] = policy_name
        tm["series_name"] = stress_policy_name
        tm["series_type"] = "active"
        tm["period"] = "test_only"
        tm.update({k: v for k, v in scenario.items() if k != "scenario_name"})
        test_metrics.append(tm)

    for benchmark_name, benchmark_w in benchmark_weights.items():
        stress_policy_name = f"{scenario_name}__{benchmark_name}"

        returns_df, _ = backtest_policy(
            policy_name=stress_policy_name,
            signal_weight_df=benchmark_w,
            etf_returns=etf_returns,
            config=scenario,
        )

        grp_test = returns_df.loc[returns_df.index.intersection(test_dates)].copy()
        if grp_test.empty:
            continue

        grp_test = rebase_return_df(grp_test)
        tm = performance_metrics(grp_test)
        tm["run_id"] = RUN_ID
        tm["scenario_name"] = scenario_name
        tm["base_policy_name"] = benchmark_name
        tm["series_name"] = stress_policy_name
        tm["series_type"] = "benchmark"
        tm["period"] = "test_only"
        tm.update({k: v for k, v in scenario.items() if k != "scenario_name"})
        test_metrics.append(tm)

test_metrics_df = pd.DataFrame(test_metrics)

test_metrics_df.to_csv(TABLE_RUN_DIR / "stress_test_metrics_test_only.csv", index=False)
test_metrics_df.to_parquet(TABLE_RUN_DIR / "stress_test_metrics_test_only.parquet", index=False)
test_metrics_df.to_csv(TABLE_DIR / f"table_36_stress_test_metrics_test_only_{RUN_ID}.csv", index=False)

print("Test metrics shape:", test_metrics_df.shape)

# ============================================================
# 10. Summaries and rankings
# ============================================================

print("\n" + "=" * 80)
print("Step 5: Creating stress-test rankings and summaries")
print("=" * 80)

def rank_metrics(df, period_label):
    out = df.copy()

    out["rank_total_return"] = out["total_return"].rank(ascending=False, method="min")
    out["rank_sharpe"] = out["sharpe_ratio"].rank(ascending=False, method="min")
    out["rank_sortino"] = out["sortino_ratio"].rank(ascending=False, method="min")
    out["rank_drawdown"] = out["max_drawdown"].rank(ascending=False, method="min")
    out["rank_calmar"] = out["calmar_ratio"].rank(ascending=False, method="min")

    out["stress_composite_rank"] = (
        out["rank_total_return"]
        + out["rank_sharpe"]
        + out["rank_sortino"]
        + out["rank_drawdown"]
        + out["rank_calmar"]
    ) / 5.0

    out["period"] = period_label

    return out.sort_values(
        ["stress_composite_rank", "sharpe_ratio", "total_return"],
        ascending=[True, False, False],
    )

rank_full_df = rank_metrics(metrics_df, "full_period")
rank_test_df = rank_metrics(test_metrics_df, "test_only")

rank_full_df.to_csv(TABLE_RUN_DIR / "stress_test_rankings_full_period.csv", index=False)
rank_full_df.to_csv(TABLE_DIR / f"table_37_stress_test_rankings_full_period_{RUN_ID}.csv", index=False)

rank_test_df.to_csv(TABLE_RUN_DIR / "stress_test_rankings_test_only.csv", index=False)
rank_test_df.to_csv(TABLE_DIR / f"table_38_stress_test_rankings_test_only_{RUN_ID}.csv", index=False)

# Scenario best active policy.
best_active_full = (
    rank_full_df[rank_full_df["series_type"] == "active"]
    .sort_values(["scenario_name", "stress_composite_rank"])
    .groupby("scenario_name", as_index=False)
    .first()
)

best_active_test = (
    rank_test_df[rank_test_df["series_type"] == "active"]
    .sort_values(["scenario_name", "stress_composite_rank"])
    .groupby("scenario_name", as_index=False)
    .first()
)

best_active_full.to_csv(TABLE_RUN_DIR / "best_active_policy_by_scenario_full.csv", index=False)
best_active_test.to_csv(TABLE_RUN_DIR / "best_active_policy_by_scenario_test.csv", index=False)

best_active_full.to_csv(TABLE_DIR / f"table_39_best_active_policy_by_scenario_full_{RUN_ID}.csv", index=False)
best_active_test.to_csv(TABLE_DIR / f"table_40_best_active_policy_by_scenario_test_{RUN_ID}.csv", index=False)

# Compare primary active policy vs B1 benchmark scenario by scenario.
primary_vs_b1 = []

for scenario_name in metrics_df["scenario_name"].unique():
    sub = metrics_df[metrics_df["scenario_name"] == scenario_name]

    p4 = sub[sub["base_policy_name"] == PRIMARY_POLICY]
    b1 = sub[sub["base_policy_name"] == "B1_equal_weight_all_etfs"]

    if p4.empty or b1.empty:
        continue

    p4 = p4.iloc[0]
    b1 = b1.iloc[0]

    primary_vs_b1.append({
        "scenario_name": scenario_name,
        "primary_policy": PRIMARY_POLICY,
        "primary_total_return": p4["total_return"],
        "b1_total_return": b1["total_return"],
        "excess_total_return_vs_b1": p4["total_return"] - b1["total_return"],
        "primary_sharpe": p4["sharpe_ratio"],
        "b1_sharpe": b1["sharpe_ratio"],
        "excess_sharpe_vs_b1": p4["sharpe_ratio"] - b1["sharpe_ratio"],
        "primary_max_drawdown": p4["max_drawdown"],
        "b1_max_drawdown": b1["max_drawdown"],
        "drawdown_improvement_vs_b1": p4["max_drawdown"] - b1["max_drawdown"],
    })

primary_vs_b1_df = pd.DataFrame(primary_vs_b1)
primary_vs_b1_df.to_csv(TABLE_RUN_DIR / "primary_policy_vs_b1_by_scenario.csv", index=False)
primary_vs_b1_df.to_csv(TABLE_DIR / f"table_41_primary_policy_vs_b1_by_scenario_{RUN_ID}.csv", index=False)

print("\nTop 15 full-period stress-test rankings:")
print(rank_full_df[[
    "series_name",
    "series_type",
    "scenario_name",
    "base_policy_name",
    "total_return",
    "annual_return",
    "annual_volatility",
    "sharpe_ratio",
    "max_drawdown",
    "stress_composite_rank",
]].head(15).to_string(index=False))

print("\nTop 15 test-only stress-test rankings:")
print(rank_test_df[[
    "series_name",
    "series_type",
    "scenario_name",
    "base_policy_name",
    "total_return",
    "annual_return",
    "annual_volatility",
    "sharpe_ratio",
    "max_drawdown",
    "stress_composite_rank",
]].head(15).to_string(index=False))

print("\nPrimary policy vs B1 benchmark by scenario:")
print(primary_vs_b1_df.to_string(index=False))

# ============================================================
# 11. Paper-ready figures
# ============================================================

print("\n" + "=" * 80)
print("Step 6: Creating paper-ready figures")
print("=" * 80)

# Baseline scenario comparison.
baseline_rank = rank_full_df[rank_full_df["scenario_name"] == "S00_baseline_quarterly_10bps"].copy()
baseline_rank["series_name"] = baseline_rank["base_policy_name"]

baseline_test_rank = rank_test_df[rank_test_df["scenario_name"] == "S00_baseline_quarterly_10bps"].copy()
baseline_test_rank["series_name"] = baseline_test_rank["base_policy_name"]

plot_metric_bar(
    baseline_rank,
    metric="sharpe_ratio",
    path=FIGURE_RUN_DIR / "fig_27_baseline_full_period_sharpe_by_policy.png",
    title="Full-period Sharpe ratio by policy, baseline allocation protocol",
    higher_is_better=True,
)

plot_metric_bar(
    baseline_rank,
    metric="total_return",
    path=FIGURE_RUN_DIR / "fig_28_baseline_full_period_total_return_by_policy.png",
    title="Full-period total return by policy, baseline allocation protocol",
    higher_is_better=True,
)

plot_metric_bar(
    baseline_rank,
    metric="max_drawdown",
    path=FIGURE_RUN_DIR / "fig_29_baseline_full_period_max_drawdown_by_policy.png",
    title="Full-period max drawdown by policy, baseline allocation protocol",
    higher_is_better=True,
)

plot_metric_bar(
    baseline_test_rank,
    metric="sharpe_ratio",
    path=FIGURE_RUN_DIR / "fig_30_baseline_test_period_sharpe_by_policy.png",
    title="Test-period Sharpe ratio by policy, baseline allocation protocol",
    higher_is_better=True,
)

# Stress scenario heatmaps for active policies.
active_full = rank_full_df[rank_full_df["series_type"] == "active"].copy()

plot_scenario_heatmap(
    active_full,
    row_col="scenario_name",
    col_col="base_policy_name",
    value_col="sharpe_ratio",
    path=FIGURE_RUN_DIR / "fig_31_active_policy_sharpe_heatmap_full_period.png",
    title="Active policy Sharpe ratio across stress-test scenarios",
)

plot_scenario_heatmap(
    active_full,
    row_col="scenario_name",
    col_col="base_policy_name",
    value_col="max_drawdown",
    path=FIGURE_RUN_DIR / "fig_32_active_policy_drawdown_heatmap_full_period.png",
    title="Active policy max drawdown across stress-test scenarios",
)

# 00881 cap sensitivity for primary policy.
cap_sensitivity = metrics_df[
    (metrics_df["base_policy_name"] == PRIMARY_POLICY)
    & (metrics_df["scenario_name"].isin(["S08_low_00881_cap_25pct", "S09_baseline_00881_cap_35pct", "S10_high_00881_cap_50pct"]))
].copy()

if not cap_sensitivity.empty:
    plot_sensitivity_line(
        cap_sensitivity,
        x_col="max_00881_weight",
        y_col="sharpe_ratio",
        hue_col="base_policy_name",
        path=FIGURE_RUN_DIR / "fig_33_00881_cap_sensitivity_primary_policy.png",
        title="00881 cap sensitivity for primary AURORA policy",
    )

# Cash boost sensitivity for primary policy.
cash_sensitivity = metrics_df[
    (metrics_df["base_policy_name"] == PRIMARY_POLICY)
    & (metrics_df["scenario_name"].isin(["S05_no_uncertainty_cash_boost", "S06_low_cash_boost_10pct", "S00_baseline_quarterly_10bps", "S07_high_cash_boost_30pct"]))
].copy()

if not cash_sensitivity.empty:
    plot_sensitivity_line(
        cash_sensitivity,
        x_col="uncertainty_cash_boost_max",
        y_col="sharpe_ratio",
        hue_col="base_policy_name",
        path=FIGURE_RUN_DIR / "fig_34_cash_boost_sensitivity_primary_policy.png",
        title="Uncertainty cash-boost sensitivity for primary AURORA policy",
    )

# Blend sensitivity.
blend_sensitivity = metrics_df[
    (metrics_df["base_policy_name"] == PRIMARY_POLICY)
    & (metrics_df["scenario_name"].isin(["S13_more_20d_alpha_80_20", "S14_equal_alpha_50_50", "S00_baseline_quarterly_10bps", "S15_more_60d_alpha_30_70"]))
].copy()

if not blend_sensitivity.empty:
    plot_sensitivity_line(
        blend_sensitivity,
        x_col="alpha_20d",
        y_col="sharpe_ratio",
        hue_col="base_policy_name",
        path=FIGURE_RUN_DIR / "fig_35_alpha20d_blend_sensitivity_primary_policy.png",
        title="20d/60d blend sensitivity for primary AURORA policy",
    )

# Template variant comparison.
template_sensitivity = metrics_df[
    (metrics_df["base_policy_name"] == PRIMARY_POLICY)
    & (metrics_df["scenario_name"].isin(["S00_baseline_quarterly_10bps", "S16_aggressive_templates", "S17_defensive_templates", "S18_less_defensive_combined"]))
].copy()

if not template_sensitivity.empty:
    template_sensitivity["series_name"] = template_sensitivity["scenario_name"]
    plot_metric_bar(
        template_sensitivity,
        metric="sharpe_ratio",
        path=FIGURE_RUN_DIR / "fig_36_template_sensitivity_primary_policy_sharpe.png",
        title="Template aggressiveness sensitivity for primary AURORA policy",
        higher_is_better=True,
    )

# Weight summary.
weight_summary_primary = feature_summary_df[
    feature_summary_df["base_policy_name"] == PRIMARY_POLICY
].copy()

if not weight_summary_primary.empty:
    weight_summary_primary["series_name"] = weight_summary_primary["scenario_name"]
    plot_metric_bar(
        weight_summary_primary,
        metric="mean_cash_weight",
        path=FIGURE_RUN_DIR / "fig_37_primary_policy_mean_cash_weight_by_scenario.png",
        title="Mean cash weight by stress-test scenario for primary AURORA policy",
        higher_is_better=False,
    )

# Equity curves for baseline and less defensive combined.
selected_returns_df = pd.concat(all_returns_selected, axis=0) if all_returns_selected else pd.DataFrame()

if not selected_returns_df.empty:
    plot_equity_curves(
        selected_returns_df,
        path=FIGURE_RUN_DIR / "fig_38_selected_policy_equity_curves.png",
        title="Selected AURORA stress-test equity curves",
    )

    plot_drawdowns(
        selected_returns_df,
        path=FIGURE_RUN_DIR / "fig_39_selected_policy_drawdowns.png",
        title="Selected AURORA stress-test drawdowns",
    )

# Copy paper figures to global figure dir with run id.
for fig_path in sorted(FIGURE_RUN_DIR.glob("*.png")):
    target_path = FIGURE_DIR / f"{fig_path.stem}_{RUN_ID}.png"
    target_path.write_bytes(fig_path.read_bytes())

print("Paper figures saved to:", FIGURE_RUN_DIR)

# ============================================================
# 12. Paper-ready concise tables
# ============================================================

print("\n" + "=" * 80)
print("Step 7: Creating paper-ready concise tables")
print("=" * 80)

# Table: Baseline allocation results, compact.
baseline_table = baseline_rank[[
    "base_policy_name",
    "series_type",
    "total_return",
    "annual_return",
    "annual_volatility",
    "sharpe_ratio",
    "sortino_ratio",
    "max_drawdown",
    "calmar_ratio",
    "final_equity",
    "stress_composite_rank",
]].copy()

baseline_table = baseline_table.rename(columns={"base_policy_name": "policy_name"})
baseline_table.to_csv(TABLE_RUN_DIR / "paper_table_baseline_allocation_results.csv", index=False)
baseline_table.to_csv(TABLE_DIR / f"table_42_paper_baseline_allocation_results_{RUN_ID}.csv", index=False)

# Table: Stress result for primary active policy.
primary_stress_table = metrics_df[
    metrics_df["base_policy_name"] == PRIMARY_POLICY
][[
    "scenario_name",
    "total_return",
    "annual_return",
    "annual_volatility",
    "sharpe_ratio",
    "sortino_ratio",
    "max_drawdown",
    "calmar_ratio",
    "final_equity",
    "template_variant",
    "rebalance_frequency",
    "transaction_cost_rate",
    "uncertainty_cash_boost_max",
    "max_00881_weight",
    "alpha_20d",
    "alpha_60d",
]].copy()

primary_stress_table.to_csv(TABLE_RUN_DIR / "paper_table_primary_policy_stress_tests.csv", index=False)
primary_stress_table.to_csv(TABLE_DIR / f"table_43_paper_primary_policy_stress_tests_{RUN_ID}.csv", index=False)

# Table: Best active scenario vs benchmarks.
best_active_compact = best_active_full[[
    "scenario_name",
    "base_policy_name",
    "total_return",
    "annual_return",
    "annual_volatility",
    "sharpe_ratio",
    "max_drawdown",
    "final_equity",
    "stress_composite_rank",
]].copy()

best_active_compact.to_csv(TABLE_RUN_DIR / "paper_table_best_active_policy_by_scenario.csv", index=False)
best_active_compact.to_csv(TABLE_DIR / f"table_44_paper_best_active_policy_by_scenario_{RUN_ID}.csv", index=False)

# Table: Primary vs B1.
primary_vs_b1_df.to_csv(TABLE_RUN_DIR / "paper_table_primary_vs_b1.csv", index=False)
primary_vs_b1_df.to_csv(TABLE_DIR / f"table_45_paper_primary_vs_b1_{RUN_ID}.csv", index=False)

print("\nPaper baseline allocation table:")
print(baseline_table.to_string(index=False))

print("\nPrimary policy stress-test table:")
print(primary_stress_table.to_string(index=False))

# ============================================================
# 13. Interpretation aid: diagnose likely causes
# ============================================================

print("\n" + "=" * 80)
print("Step 8: Diagnosing likely causes of underperformance")
print("=" * 80)

diagnosis_rows = []

# Baseline active vs benchmark.
baseline_primary = metrics_df[
    (metrics_df["scenario_name"] == "S00_baseline_quarterly_10bps")
    & (metrics_df["base_policy_name"] == PRIMARY_POLICY)
].iloc[0]

baseline_b1 = metrics_df[
    (metrics_df["scenario_name"] == "S00_baseline_quarterly_10bps")
    & (metrics_df["base_policy_name"] == "B1_equal_weight_all_etfs")
].iloc[0]

diagnosis_rows.append({
    "diagnostic_question": "Does baseline AURORA beat equal-weight benchmark?",
    "finding": "No" if baseline_primary["sharpe_ratio"] < baseline_b1["sharpe_ratio"] else "Yes",
    "evidence": (
        f"Primary Sharpe={baseline_primary['sharpe_ratio']:.4f}; "
        f"B1 Sharpe={baseline_b1['sharpe_ratio']:.4f}; "
        f"Primary total return={baseline_primary['total_return']:.4f}; "
        f"B1 total return={baseline_b1['total_return']:.4f}."
    ),
})

# Cash boost effect.
no_cash = metrics_df[
    (metrics_df["scenario_name"] == "S05_no_uncertainty_cash_boost")
    & (metrics_df["base_policy_name"] == PRIMARY_POLICY)
]

baseline_cash = metrics_df[
    (metrics_df["scenario_name"] == "S00_baseline_quarterly_10bps")
    & (metrics_df["base_policy_name"] == PRIMARY_POLICY)
]

if not no_cash.empty and not baseline_cash.empty:
    no_cash = no_cash.iloc[0]
    baseline_cash = baseline_cash.iloc[0]
    diagnosis_rows.append({
        "diagnostic_question": "Does cash gating materially hurt primary AURORA performance?",
        "finding": "Yes" if no_cash["sharpe_ratio"] > baseline_cash["sharpe_ratio"] else "No",
        "evidence": (
            f"No-cash-boost Sharpe={no_cash['sharpe_ratio']:.4f}; "
            f"baseline Sharpe={baseline_cash['sharpe_ratio']:.4f}; "
            f"No-cash-boost total return={no_cash['total_return']:.4f}; "
            f"baseline total return={baseline_cash['total_return']:.4f}."
        ),
    })

# Aggressive template effect.
aggressive = metrics_df[
    (metrics_df["scenario_name"] == "S16_aggressive_templates")
    & (metrics_df["base_policy_name"] == PRIMARY_POLICY)
]

if not aggressive.empty:
    aggressive = aggressive.iloc[0]
    diagnosis_rows.append({
        "diagnostic_question": "Do more aggressive templates improve primary AURORA?",
        "finding": "Yes" if aggressive["sharpe_ratio"] > baseline_primary["sharpe_ratio"] else "No",
        "evidence": (
            f"Aggressive Sharpe={aggressive['sharpe_ratio']:.4f}; "
            f"baseline Sharpe={baseline_primary['sharpe_ratio']:.4f}; "
            f"Aggressive total return={aggressive['total_return']:.4f}; "
            f"baseline total return={baseline_primary['total_return']:.4f}."
        ),
    })

# 00881 cap effect.
high_cap = metrics_df[
    (metrics_df["scenario_name"] == "S10_high_00881_cap_50pct")
    & (metrics_df["base_policy_name"] == PRIMARY_POLICY)
]

if not high_cap.empty:
    high_cap = high_cap.iloc[0]
    diagnosis_rows.append({
        "diagnostic_question": "Does relaxing the 00881 cap improve primary AURORA?",
        "finding": "Yes" if high_cap["sharpe_ratio"] > baseline_primary["sharpe_ratio"] else "No",
        "evidence": (
            f"High-00881-cap Sharpe={high_cap['sharpe_ratio']:.4f}; "
            f"baseline Sharpe={baseline_primary['sharpe_ratio']:.4f}; "
            f"High-00881-cap total return={high_cap['total_return']:.4f}; "
            f"baseline total return={baseline_primary['total_return']:.4f}."
        ),
    })

# Blend effect.
more20 = metrics_df[
    (metrics_df["scenario_name"] == "S13_more_20d_alpha_80_20")
    & (metrics_df["base_policy_name"] == PRIMARY_POLICY)
]

more60 = metrics_df[
    (metrics_df["scenario_name"] == "S15_more_60d_alpha_30_70")
    & (metrics_df["base_policy_name"] == PRIMARY_POLICY)
]

if not more20.empty and not more60.empty:
    more20 = more20.iloc[0]
    more60 = more60.iloc[0]
    best_blend = "more 20d" if more20["sharpe_ratio"] >= more60["sharpe_ratio"] else "more 60d"
    diagnosis_rows.append({
        "diagnostic_question": "Which horizon blend is more favorable for primary AURORA?",
        "finding": best_blend,
        "evidence": (
            f"80/20 Sharpe={more20['sharpe_ratio']:.4f}; "
            f"30/70 Sharpe={more60['sharpe_ratio']:.4f}; "
            f"baseline 60/40 Sharpe={baseline_primary['sharpe_ratio']:.4f}."
        ),
    })

diagnosis_df = pd.DataFrame(diagnosis_rows)
diagnosis_df.to_csv(TABLE_RUN_DIR / "underperformance_diagnosis.csv", index=False)
diagnosis_df.to_csv(TABLE_DIR / f"table_46_underperformance_diagnosis_{RUN_ID}.csv", index=False)

print(diagnosis_df.to_string(index=False))

# ============================================================
# 14. Report and manifest
# ============================================================

print("\n" + "=" * 80)
print("Step 9: Saving validation report and manifest")
print("=" * 80)

validation_report = {
    "project_code": PROJECT_CODE,
    "notebook": "06_AURORA_allocation_stress_tests_and_paper_figures.ipynb",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "notebook05_run_id": NOTEBOOK05_RUN_ID,
    "notebook05_root": str(NOTEBOOK05_ROOT),
    "notebook05_input_index": str(NOTEBOOK05_INPUT_INDEX),
    "etf_return_panel": str(etf_return_path),
    "etf_universe": ETF_UNIVERSE,
    "primary_policy": PRIMARY_POLICY,
    "policies_to_stress": POLICIES_TO_STRESS,
    "baseline_config": BASELINE_CONFIG,
    "stress_scenarios": STRESS_SCENARIOS,
    "template_variants": {
        "baseline": BASELINE_CLASS_WEIGHT_TEMPLATES,
        "aggressive": AGGRESSIVE_CLASS_WEIGHT_TEMPLATES,
        "defensive": DEFENSIVE_CLASS_WEIGHT_TEMPLATES,
    },
    "n_stress_metric_rows": int(len(metrics_df)),
    "n_test_metric_rows": int(len(test_metrics_df)),
    "n_feature_summary_rows": int(len(feature_summary_df)),
    "diagnosis": diagnosis_df.to_dict(orient="records"),
    "educational_note": (
        "This notebook performs sensitivity analysis for a research backtest. "
        "It does not provide personalized financial advice or trading guarantees."
    ),
    "output_paths": {
        "run_root": str(RUN_ROOT),
        "tables": str(TABLE_RUN_DIR),
        "plots": str(PLOT_DIR),
        "paper_figures": str(FIGURE_RUN_DIR),
        "returns": str(RETURN_DIR),
        "weights": str(WEIGHT_DIR),
        "diagnostics": str(DIAGNOSTIC_DIR),
    },
}

validation_report_path = REPORT_RUN_DIR / "AURORA_06_stress_tests_validation_report.json"
validation_report_global_path = REPORT_DIR / f"AURORA_06_stress_tests_validation_report_{RUN_ID}.json"

save_json(validation_report_path, validation_report)
save_json(validation_report_global_path, validation_report)

manifest_df = make_file_manifest(RUN_ROOT)

manifest_path = REPORT_RUN_DIR / "AURORA_06_stress_tests_file_manifest_SHA256.csv"
manifest_global_path = REPORT_DIR / f"AURORA_06_stress_tests_file_manifest_SHA256_{RUN_ID}.csv"

manifest_df.to_csv(manifest_path, index=False)
manifest_df.to_csv(manifest_global_path, index=False)

# ============================================================
# 15. Final summary
# ============================================================

print("\n" + "=" * 80)
print("AURORA-TWETF NOTEBOOK 06 COMPLETE")
print("=" * 80)
print("Run ID                         :", RUN_ID)
print("Run root                       :", RUN_ROOT)
print("Stress metrics table           :", TABLE_DIR / f"table_33_stress_test_metrics_all_{RUN_ID}.csv")
print("Test stress metrics table      :", TABLE_DIR / f"table_36_stress_test_metrics_test_only_{RUN_ID}.csv")
print("Full-period rankings table     :", TABLE_DIR / f"table_37_stress_test_rankings_full_period_{RUN_ID}.csv")
print("Test-period rankings table     :", TABLE_DIR / f"table_38_stress_test_rankings_test_only_{RUN_ID}.csv")
print("Primary stress table           :", TABLE_DIR / f"table_43_paper_primary_policy_stress_tests_{RUN_ID}.csv")
print("Primary vs B1 table            :", TABLE_DIR / f"table_45_paper_primary_vs_b1_{RUN_ID}.csv")
print("Underperformance diagnosis     :", TABLE_DIR / f"table_46_underperformance_diagnosis_{RUN_ID}.csv")
print("Paper figures directory        :", FIGURE_RUN_DIR)
print("Validation report              :", validation_report_path)
print("Manifest                       :", manifest_path)
print("=" * 80)

print("\nRecommended next step:")
print("Send me the Notebook 06 result log. Then I will help write the final Results, Ablation, Limitations, and Conclusion sections.")

Mounted at /content/drive
AURORA-TWETF Notebook 06: Allocation Stress Tests and Paper Figures
Timestamp UTC       : 2026-06-24T01:28:18Z
Run ID              : 20260624_012818
Project root        : /content/drive/MyDrive/AURORA_TWETF
Notebook 05 root    : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/uncertainty_aware_etf_allocation/run_20260624_004516
Notebook 05 registry: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/ordinal_imbalance_uncertainty/run_20260623_151920/allocation_inputs_adjusted_before_05/NOTEBOOK05_INPUT_INDEX.csv
Run root            : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/allocation_stress_tests_and_paper_figures/run_20260624_012818

Step 1: Loading Notebook 04B registry, Notebook 05 outputs, and ETF returns
Loaded ETF return panel: /content/drive/MyDrive/AURORA_TWETF/data/panels/AURORA_etf_return_panel.parquet
ETF return shape       : (1426, 4)
ETF return date range  : 2021-01-01 to 2026-06-23

Notebook 05 registry:
           